We're going to try to implement the VAR(2) on the CTA data from say, two stations.

In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [9]:
# Look-up stations
stations = pd.read_csv("../data/clean/station-lookup.csv")

def lookup(value):
    match = stations[stations["station_id"] == value]
    if not match.empty:
        print(match.iloc[0]["stationname"])
        return

    match = stations[stations["stationname"] == value]
    if not match.empty:
        print(match.iloc[0]["station_id"])
        return
    
    print(f"No match found for: {value}")

lookup(40010)
lookup("Austin-Forest Park")

Austin-Forest Park
40010


In [10]:
cta_data_red_weekly = pd.read_csv("../data/clean/cta-data-red-weekly.csv")
selected_stations = ['s_40080', 's_40100', 's_40190', 's_40240']

cta_data_red_weekly = cta_data_red_weekly.iloc[:, 1:]
cta_data_red_weekly = cta_data_red_weekly[selected_stations]
cta_data_red_weekly = cta_data_red_weekly.to_numpy()

cta_data_red_weekly.shape[0]
cta_data_red_weekly.shape[1]

4

In [11]:
from cmdstanpy import CmdStanModel, set_cmdstan_path
set_cmdstan_path("/deac/sta/classes/sta720/software/cmdstan/2.37.0")

# To write rest of Stan model...
stan_data = {
    "T": cta_data_red_weekly.shape[0],
    "K": cta_data_red_weekly.shape[1],
    "Y": cta_data_red_weekly
}

var_mod = CmdStanModel(stan_file="var_cta_test.stan")
var_mod_post = var_mod.sample(data = stan_data)
print(var_mod_post.diagnose())
var_mod_params = var_mod_post.stan_variables()

14:21:57 - cmdstanpy - INFO - compiling stan file /deac/sta/classes/sta720-sp-2026/hatan25/sta720-project/notebooks/var_cta_test.stan to exe file /deac/sta/classes/sta720-sp-2026/hatan25/sta720-project/notebooks/var_cta_test
14:22:30 - cmdstanpy - INFO - compiled model executable: /deac/sta/classes/sta720-sp-2026/hatan25/sta720-project/notebooks/var_cta_test
14:22:30 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

14:29:50 - cmdstanpy - INFO - CmdStan done processing.
14:29:50 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, column 4 to column 22)
	Exception: lkj_corr_lpdf: Correlation matrix is not positive definite. (in 'var_cta_test.stan', line 22, 


Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.



In [20]:
var_mod_params["A_2"]

array([[[-0.19146359,  0.12677203,  0.02306113, -0.07790664],
        [ 0.00630749,  0.00679226,  0.00953523, -0.13034264],
        [-0.00497315,  0.35817534, -0.32261866, -0.14327123],
        [-0.00180191,  0.23068263,  0.01948508, -0.3078385 ]],

       [[-0.0647254 , -0.25851102,  0.05459032,  0.04363622],
        [ 0.10594671, -0.24975069,  0.02399147, -0.06483444],
        [ 0.02297535,  0.17975408, -0.33067659, -0.04886444],
        [ 0.03543349,  0.01340357,  0.02962512, -0.21809388]],

       [[-0.26177164,  0.03586776,  0.05438176,  0.06278864],
        [-0.00546506, -0.15004687,  0.00622361,  0.03320455],
        [ 0.04755793,  0.04798132, -0.27567238,  0.00843016],
        [-0.02209772,  0.08027064,  0.01972334, -0.14616311]],

       ...,

       [[-0.17511163, -0.03643182,  0.08722292, -0.07763257],
        [ 0.0048516 , -0.12240991,  0.04198409, -0.06507321],
        [ 0.03502357,  0.22117331, -0.27347861, -0.09377042],
        [-0.0246707 ,  0.20604804,  0.04548656, -0.